In [2]:
# Iniciando una session en Spark
# Importar la clase principal para crear la sesión de Spark
from pyspark.sql import SparkSession
ruta = "../documentos/"


# Crear o recuperar una sesión de Spark
# appName es opcional pero se recomienda para identificar el job
spark = SparkSession.builder \
    .appName("mi_app_pyspark") \
    .getOrCreate()

sc = spark.sparkContext # con esto podemos usar el contexto de spark para crear RDDs

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/17 20:31:10 WARN Utils: Your hostname, MacBook-Air-de-Fernando.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.120 instead (on interface en0)
26/03/17 20:31:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 20:31:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Funcion map

In [3]:
rdd = sc.parallelize([1,2,3,5,7,11,13,17,19])

In [4]:
rdd_resta = rdd.map(lambda x: x-1)
print(rdd_resta.collect())

[0, 1, 2, 4, 6, 10, 12, 16, 18]


In [5]:
rdd_par = rdd.map(lambda x: x % 2 == 0)
rdd_par.collect()

[False, True, False, False, False, False, False, False, False]

In [6]:
rdd_texto = sc.parallelize(["hola", "mundo", "pyspark"])
rdd_longitud = rdd_texto.map(lambda x: len(x))
print(rdd_longitud.collect())

[4, 5, 7]


In [7]:
rdd_mayusculas = rdd_texto.map(lambda x: x.upper())
print(rdd_mayusculas.collect())

['HOLA', 'MUNDO', 'PYSPARK']


In [8]:
rdd_hola = rdd_texto.map(lambda x: 'hola ' + x)
print(rdd_hola.collect())

['hola hola', 'hola mundo', 'hola pyspark']


### Funcion flatMap


In [16]:
rdd = sc.parallelize([1,2,4,6,8,3,5,7,11,13,17,19])


In [11]:
rdd_cuadrado = rdd.map(lambda x: (x, x**2))
print(rdd_cuadrado.collect()) #Con map lo que noos devuelve es una dupla, con el elemento y su cuadrado

[(1, 1), (2, 4), (3, 9), (5, 25), (7, 49), (11, 121), (13, 169), (17, 289), (19, 361)]


In [13]:
# Si quieremos una lista rn lugar de una dupla, podemos usamos flatMap
rdd_cuadrado_flat = rdd.flatMap(lambda x: (x, x**2))
print(rdd_cuadrado_flat.collect()) #Ya no nos devuelve las duclas, sino una lista con los elementos

[1, 1, 2, 4, 3, 9, 5, 25, 7, 49, 11, 121, 13, 169, 17, 289, 19, 361]


In [14]:
rdd_texto = sc.parallelize(["jose", "juan", "maria", "ana", "pedro"])
rdd_mayuscula = rdd_texto.flatMap(lambda x: x.upper())
print(rdd_mayuscula.collect()) #Con flatMap nos devuelve cada letra por separado

['J', 'O', 'S', 'E', 'J', 'U', 'A', 'N', 'M', 'A', 'R', 'I', 'A', 'A', 'N', 'A', 'P', 'E', 'D', 'R', 'O']


### Filter
Genera particiones filtradas de salida

In [17]:
rdd_par = rdd.filter(lambda x: x % 2 == 0)
print(rdd_par.collect())

[2, 4, 6, 8]


In [18]:
rdd_impares = rdd.filter(lambda x: x % 2 != 0)
print(rdd_impares.collect())

[1, 3, 5, 7, 11, 13, 17, 19]


In [19]:
rdd_k = rdd_texto.filter(lambda x: len(x) > 4)
print(rdd_k.collect())

['maria', 'pedro']


In [20]:
rdd_ini = rdd_texto.filter(lambda x: x.startswith('j'))
print(rdd_ini.collect())

['jose', 'juan']


In [ ]:
rdd_filtro = rdd_texto.filter(lambda x: x.startswith('j') and x.find('u')== 1) # Busca palabras que empiecen por j y tengan una u en la segunda posición
print(rdd_filtro.collect())

['juan']


### Transformacion coalesce

In [23]:
rdd = sc.parallelize([1,2,3,4,10,60,5,7,11,13,17,19], 10) # Creamos un RDD con 4 particiones
print(rdd.getNumPartitions()) # Nos devuelve el número de particiones que tiene el R

10


In [24]:
rdd.coalesce(5)
rdd.getNumPartitions() # Nos devuelve el número de particiones que tiene el RDD después de coalesce

10

In [25]:
rdd5 = rdd.coalesce(5)
print(rdd5.getNumPartitions()) # Nos devuelve el número de particiones que tiene el

5


### Repartition
Redistribuye las particiones, ya sea juntandoles o dividiendoles, para disminuir o aumentar particiones

In [27]:
rdd = sc.parallelize([1,2,3,5,7,11,13,17,19], 3)
rdd.getNumPartitions() # Nos devuelve el número de particiones que tiene el RDD

3

In [28]:
rdd7 = rdd.repartition(7)
print(rdd7.getNumPartitions()) # Nos devuelve el número de particiones que tiene el

7


## coalesce VS repartition

**`coalesce`** se usa solo para **reducir** el número de particiones. Esta es una versión optimizada de `repartition` donde el movimiento de los datos a través de las particiones (shuffle) es menor.

> ### ⚠️ Importante
> `repartition` y `coalesce` son operaciones muy costosas ya que mezclan los datos en muchas particiones; por lo tanto, intente minimizar el uso de estos tanto como sea posible.

### Reduce by Key

In [30]:
### Reduce by Key
rdd = sc.parallelize(
    [('casa', 2),
     ('parque', 1),
     ('que', 5),
     ('casa', 1),
     ('escuela', 2),
     ('casa', 1),
     ('que', 1)]
)

In [31]:
print(rdd.collect())

[('casa', 2), ('parque', 1), ('que', 5), ('casa', 1), ('escuela', 2), ('casa', 1), ('que', 1)]


In [ ]:
rdd_reducido = rdd.reduceByKey(lambda x,y: x+y)
print(rdd_reducido.collect()) #Va llave por llave y va sumando los valores asociados a cada llave, en este caso las palabras y el número de veces que aparecen.

[('casa', 4), ('parque', 1), ('escuela', 2), ('que', 6)]


## Acciones

# Concepto: Lazy Evaluation en Apache Spark

La **Evaluación Perezosa (Lazy Evaluation)** significa que Spark no ejecuta las transformaciones inmediatamente. En su lugar, construye un grafo de ejecución (DAG) y solo procesa los datos cuando se invoca una **Acción**.

### Flujo de Trabajo del RDD

| Lógica del Proceso | Variable RDD | Código de PySpark | Tipo de Operación |
| :--- | :--- | :--- | :--- |
| **RDD Original** | `rdd` | `rdd = sc.parallelize(['azul', 'azul', 'verde', 'rojo', 'rojo', 'verde', 'negro'])` | Creación |
| **Filtrar los azules** | `sin_azul_rdd` | `sin_azul_rdd = rdd.filter(lambda x: x != 'azul')` | Transformación |
| **Filtrar los rojos** | `sin_rojo_rdd` | `sin_rojo_rdd = sin_azul_rdd.filter(lambda x: x != 'rojo')` | Transformación |
| **Mostrar registros** | *(Resultado)* | `sin_rojo_rdd.count()` | **Acción** ⚡ |
| **Tomar solo negros** | `solo_negro_rdd` | `solo_negro_rdd = sin_rojo_rdd.filter(lambda x: x == 'negro')` | Transformación |
| **Mostrar registros** | *(Resultado)* | `solo_negro_rdd.count()` | **Acción** ⚡ |

---

### Bloque de Código Completo

A continuación se muestra la implementación siguiendo el flujo del diagrama:

```python
# 1. Creación del RDD
rdd = sc.parallelize(['azul', 'azul', 'verde', 'rojo', 'rojo', 'verde', 'negro'])

# 2. Transformaciones (No se ejecutan todavía, solo se planifican)
sin_azul_rdd = rdd.filter(lambda x: x != 'azul')
sin_rojo_rdd = sin_azul_rdd.filter(lambda x: x != 'rojo')

# 3. Primera Acción (Aquí es donde Spark realmente trabaja)
print(f"Conteo después de filtrar rojos: {sin_rojo_rdd.count()}")

# 4. Más transformaciones sobre el linaje existente
solo_negro_rdd = sin_rojo_rdd.filter(lambda x: x == 'negro')

# 5. Segunda Acción
print(f"Conteo de negros: {solo_negro_rdd.count()}")

### Reduce

In [33]:
rdd = sc.parallelize([1,2,3,5,7,11,13,17,19])


In [34]:
rdd.reduce(lambda x,y: x+y) # Nos devuelve la suma de todos los elementos del RDD, en este caso 77


78

In [35]:
rdd1 = sc.parallelize([1,2,3,4,5])


In [36]:
rdd1.reduce(lambda x,y: x*y)

120

### Count
cuanta el numero de elementos del rdd

In [37]:
rdd_texto = sc.parallelize(["hola", "mundo", "pyspark"])


In [38]:
rdd_texto.count() # Nos devuelve el número de elementos que hay en el RDD, en este caso 3

3

### Funcuones take, max y saveAsTextFile

In [39]:
rdd = sc.parallelize('La programacion es bella'.split(' '))
print(rdd.collect())

['La', 'programacion', 'es', 'bella']


In [40]:
rdd.take(3) # Nos devuelve los 3 primeros elementos del RDD, en este caso ['La', 'programacion', 'es']

['La', 'programacion', 'es']

In [42]:
rdd1 = sc.parallelize([item / (item + 1) for item in range(10)])
rdd1.max()

0.9